In [3]:
!pip install ultralytics fastapi python-multipart pillow numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [4]:
from fastapi import FastAPI, UploadFile, File
from fastapi.testclient import TestClient
from PIL import Image
from ultralytics import YOLO
import numpy as np
import io

app = FastAPI()

# Load AI model
model = YOLO("yolov8n.pt")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    # STEP 1: API INPUT
    image_bytes = await file.read()
    print("1. API Input: Image received")

    # STEP 2: PREPROCESSING
    image = Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

    image = image.resize((640, 640))
    image_array = np.array(image)

    print("2. Preprocessing: Image resized")

    # STEP 3: MODEL EXECUTION
    results = model.predict(
        source=image_array,
        conf=0.25,
        verbose=False
    )

    print("3. Model Execution: YOLO completed")

    # STEP 4: RESPONSE
    detections = []

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append({
            "object": model.names[class_id],
            "confidence": round(confidence * 100, 2)
        })

    print("4. Response: Results returned")

    return {
        "filename": file.filename,
        "image_size": [640, 640],
        "detections": detections
    }

print("Inference Request Trace Ready!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Inference Request Trace Ready!


In [6]:
client = TestClient(app)

In [7]:
from google.colab import files

uploaded = files.upload()

image_name = list(uploaded.keys())[0]

with open(image_name, "rb") as f:
    response = client.post(
        "/predict",
        files={
            "file": (
                image_name,
                f,
                "image/jpeg"
            )
        }
    )

print("Status Code:", response.status_code)
print("Final Response:", response.json())

Saving Screenshot 2026-11-13 121154.png to Screenshot 2026-11-13 121154.png
1. API Input: Image received
2. Preprocessing: Image resized
3. Model Execution: YOLO completed
4. Response: Results returned
Status Code: 200
Final Response: {'filename': 'Screenshot 2026-11-13 121154.png', 'image_size': [640, 640], 'detections': []}
